# WiniCari -- 07 Évaluation des Modèles

Évaluation de bout en bout des quatre modules IA sur des **données de test non vues**.

| Module | Modèles | Méthode d'évaluation |
|--------|---------|----------------------|
| 1 Retard | HistGBM + LSTM + Prophet | Split 80/20 chronologique par jour ; MAE / RMSE / R2 |
| 2 Repli GPS | Kalman + correction LSTM | Masquage synthétique d'écarts de 3 min ; erreur médiane / P90 |
| 3 Anomalie | Isolation Forest + LSTM AE | Distributions des scores ; séparation des caractéristiques ; accord |
| 4 Chatbot RAG | ChromaDB + Llama 3 | Test de requêtes en direct avec inspection du contexte |

## 0. Configuration

In [55]:
from pathlib import Path
import sys, os
sys.path.insert(0, str(Path.cwd().parent))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from dotenv import load_dotenv
load_dotenv("../.env")

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})

FOUNDATION = Path("../data/processed/foundation_arrivals_full.parquet")
LINE_DIST  = Path("../data/processed/line_distances.parquet")
MODELS_DIR = Path("../models")

print("Imports OK")
print(f"Foundation : {FOUNDATION.exists()}")
print(f"Models dir : {MODELS_DIR.exists()}")


Imports OK
Foundation : True
Models dir : True


---
## 1. Module 1 -- Prédiction de Retard

### 1.1 Charger les modèles et reconstruire le split de test

Reproduction du même split chronologique 80/20 par jour utilisé pendant l'entraînement — tout avec `day >= cut_day` constitue les données de test non vues.

In [56]:
from src.models import delay as delay_mod
from src.data import delay as dl

models_delay = delay_mod.load(MODELS_DIR / "delay")

cfg  = dl.DelayConfig()
df   = dl.load_foundation(FOUNDATION)
m    = dl.add_daytype(dl.with_elapsed(df, cfg))
base = dl.build_baseline(m, cfg)
d    = dl.add_daytype(dl.with_delay(m, base, cfg))
roll = dl.rolling_table(d)

days    = np.sort(roll["day"].unique())
cut_day = days[int(0.8 * len(days))]
te = roll[roll["day"] >= cut_day].copy()
tr = roll[roll["day"]  < cut_day].copy()

print(f"Train days : {tr['day'].nunique()}  ({len(tr):,} rows)")
print(f"Test  days : {te['day'].nunique()}  ({len(te):,} rows)  from day {cut_day}")


Delay models loaded: HistGBM + LSTM + 33 Prophet models
Train days : 416  (95,225 rows)
Test  days : 104  (12,009 rows)  from day 20260302


### 1.2 Comparaison des métriques -- MAE, RMSE, R2

In [57]:
from src.data.delay import build_lstm_sequences, scale_sequences

X, _, y_seq = build_lstm_sequences(roll)
day_arr = roll["day"].values

X_te_raw = X[day_arr >= cut_day]
y_te     = y_seq[day_arr >= cut_day]

feat_mean = models_delay["feat_mean"]
feat_std  = models_delay["feat_std"]
X_te_s    = scale_sequences(X_te_raw, feat_mean, feat_std)

pred_hgbm  = models_delay["hgbm"].predict(dl._design(te))
pred_lstm   = dl.predict_lstm(models_delay["lstm"], X_te_s)
pred_pers   = te["delay_min"].values
pred_naive  = np.zeros(len(te))

y_hgbm = te[dl.TARGET].values

def _m(y_true, y_pred, name):
    return {"Model": name,
            "MAE (min)":  round(mean_absolute_error(y_true, y_pred), 2),
            "RMSE (min)": round(mean_squared_error(y_true, y_pred)**0.5, 2),
            "R2":         round(r2_score(y_true, y_pred), 3)}

metric_df = pd.DataFrame([
    _m(y_hgbm, pred_naive, "Naive (always 0)"),
    _m(y_hgbm, pred_pers,  "Persistence (next = current)"),
    _m(y_hgbm, pred_hgbm, "HistGBM"),
    _m(y_te,   pred_lstm,  "LSTM"),
])
print("Delay Prediction -- Test Set Metrics")
display(metric_df.to_string(index=False))


Delay Prediction -- Test Set Metrics


'                       Model  MAE (min)  RMSE (min)     R2\n            Naive (always 0)      13.76       23.07 -0.003\nPersistence (next = current)       3.06        7.66  0.889\n                     HistGBM       2.75        6.82  0.912\n                        LSTM       3.03        7.12  0.904'

### 1.3 Distributions des erreurs et prédit vs réel

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Prédiction de Retard -- Diagnostics sur l'Ensemble de Test", fontsize=13, fontweight="bold")

# A: Graphique MAE en barres
ax = axes[0, 0]
colors = ["#d9d9d9", "#bdbdbd", "#4393c3", "#d6604d"]
bars = ax.bar(metric_df["Model"], metric_df["MAE (min)"], color=colors, edgecolor="white")
ax.set_title("A -- Comparaison MAE (plus bas = meilleur)")
ax.set_ylabel("MAE (minutes)")
ax.set_xticks(range(len(metric_df)))
ax.set_xticklabels(metric_df["Model"], rotation=20, ha="right")
for b, v in zip(bars, metric_df["MAE (min)"]):
    ax.text(b.get_x() + b.get_width()/2, v + 0.05, f"{v:.2f}", ha="center", fontsize=9)

# B: Histogramme des résidus
ax = axes[0, 1]
bins = np.linspace(-25, 25, 60)
ax.hist(y_hgbm - pred_hgbm, bins=bins, alpha=0.6, color="#4393c3", label="HistGBM")
ax.hist(y_te   - pred_lstm,  bins=bins, alpha=0.6, color="#d6604d", label="LSTM")
ax.axvline(0, color="black", lw=1.2, ls="--")
ax.set_title("B -- Distribution des résidus")
ax.set_xlabel("Réel - Prédit (min)"); ax.set_ylabel("Nombre"); ax.legend()

# C: Prédit vs Réel -- HistGBM
ax = axes[1, 0]
lim = 30
idx = np.random.default_rng(0).choice(len(y_hgbm), min(3000, len(y_hgbm)), replace=False)
ax.scatter(y_hgbm[idx], pred_hgbm[idx], alpha=0.15, s=8, color="#4393c3")
ax.plot([-lim, lim], [-lim, lim], "k--", lw=1)
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
ax.set_xlabel("Retard réel (min)"); ax.set_ylabel("Prédit (min)")
ax.set_title(f"C -- HistGBM  (MAE={metric_df.loc[2,'MAE (min)']:.2f} min)")

# D: Prédit vs Réel -- LSTM
ax = axes[1, 1]
idx2 = np.random.default_rng(1).choice(len(y_te), min(3000, len(y_te)), replace=False)
ax.scatter(y_te[idx2], pred_lstm[idx2], alpha=0.15, s=8, color="#d6604d")
ax.plot([-lim, lim], [-lim, lim], "k--", lw=1)
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
ax.set_xlabel("Retard réel (min)"); ax.set_ylabel("Prédit (min)")
ax.set_title(f"D -- LSTM  (MAE={metric_df.loc[3,'MAE (min)']:.2f} min)")

plt.tight_layout(); plt.show()

### 1.4 Décomposition des erreurs -- par heure de départ et type de jour

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

te2 = te.copy()
te2["err_hgbm"] = np.abs(y_hgbm - pred_hgbm)
n_lstm_te = len(y_te)
te2["err_lstm"] = np.nan
te2.iloc[:n_lstm_te, te2.columns.get_loc("err_lstm")] = np.abs(y_te - pred_lstm)

by_h = te2.groupby("dep_hour")[["err_hgbm", "err_lstm"]].mean()
axes[0].plot(by_h.index, by_h["err_hgbm"], "o-", color="#4393c3", label="HistGBM")
axes[0].plot(by_h.index, by_h["err_lstm"],  "s-", color="#d6604d", label="LSTM")
axes[0].set_title("MAE par heure de départ")
axes[0].set_xlabel("Heure du jour"); axes[0].set_ylabel("MAE (min)"); axes[0].legend()
axes[0].set_xticks(range(0, 24, 2))

te2["day_type"] = te2["is_weekend"].map({0: "Semaine", 1: "Weekend"})
by_dt = te2.groupby("day_type")[["err_hgbm", "err_lstm"]].mean()
x = np.arange(len(by_dt)); w = 0.35
axes[1].bar(x - w/2, by_dt["err_hgbm"], w, color="#4393c3", label="HistGBM")
axes[1].bar(x + w/2, by_dt["err_lstm"],  w, color="#d6604d", label="LSTM")
axes[1].set_xticks(x); axes[1].set_xticklabels(by_dt.index)
axes[1].set_title("MAE par type de jour"); axes[1].set_ylabel("MAE (min)"); axes[1].legend()

by_line = te2.groupby("line")[["err_hgbm", "err_lstm"]].mean().sort_values("err_hgbm", ascending=False)
top = by_line.head(10)
y_pos = np.arange(len(top))
axes[2].barh(y_pos - 0.2, top["err_hgbm"], 0.35, color="#4393c3", label="HistGBM")
axes[2].barh(y_pos + 0.2, top["err_lstm"],  0.35, color="#d6604d", label="LSTM")
axes[2].set_yticks(y_pos); axes[2].set_yticklabels(top.index)
axes[2].set_title("MAE par ligne (pires 10)"); axes[2].set_xlabel("MAE (min)"); axes[2].legend()

plt.tight_layout(); plt.show()

### 1.5 Démo ETA -- simulation bus en direct

In [ ]:
test_line = te["line"].value_counts().index[0]
test_soc  = te[te["line"] == test_line]["societe"].iloc[0]
test_dir  = te[te["line"] == test_line]["dir"].iloc[0]
dep_time  = "2026-06-15 06:00"

print(f"Démo : ligne={test_line}  dir={test_dir}  compagnie={test_soc}")

eta_hgbm = delay_mod.predict_eta(models_delay, societe=test_soc, line=test_line,
                                   direction=test_dir, dep_time=dep_time,
                                   current_seq=3, current_delay_min=5.0, model_type="hgbm")
eta_lstm  = delay_mod.predict_eta(models_delay, societe=test_soc, line=test_line,
                                   direction=test_dir, dep_time=dep_time,
                                   current_seq=3, current_delay_min=5.0, model_type="lstm")

fig, ax = plt.subplots(figsize=(10, 4))
if len(eta_hgbm):
    ax.plot(eta_hgbm["seq"], eta_hgbm["pred_delay_min"], "o--", color="#4393c3",
            label="ETA HistGBM", lw=1.5)
if len(eta_lstm):
    ax.plot(eta_lstm["seq"],  eta_lstm["pred_delay_min"],  "s-",  color="#d6604d",
            label="ETA LSTM",    lw=1.5)
ax.axhline(5.0, color="grey", ls=":", lw=1.2, label="Retard actuel (+5 min)")
ax.set_xlabel("Indice d'arrêt"); ax.set_ylabel("Retard prédit (min)")
ax.set_title(f"Prévision ETA -- ligne {test_line} {test_dir}")
ax.legend(); plt.tight_layout(); plt.show()

if len(eta_hgbm) and len(eta_lstm):
    merged = (eta_hgbm[["seq","expected_min","pred_delay_min"]]
              .rename(columns={"pred_delay_min":"hgbm_delay"})
              .merge(eta_lstm[["seq","pred_delay_min"]]
                     .rename(columns={"pred_delay_min":"lstm_delay"}), on="seq"))
    print("Tableau ETA (8 premiers arrêts) :")
    display(merged.head(8))

### 1.6 Prophet -- prévision de retard saisonnière

In [ ]:
avail = list(models_delay["prophet"].keys())
key   = next((k for k in avail if "209" in k), avail[0])
pm    = models_delay["prophet"][key]
print(f"Modèle Prophet : {key}")

future = pm.make_future_dataframe(periods=30)
fc     = pm.predict(future)

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(fc["ds"], fc["yhat"], color="steelblue", lw=1, label="Ajusté + prévision")
ax.fill_between(fc["ds"], fc["yhat_lower"], fc["yhat_upper"], alpha=0.15, color="steelblue")
ax.axvspan(fc["ds"].iloc[-31], fc["ds"].iloc[-1], alpha=0.06, color="crimson", label="Période de prévision")
ax.set_title(f"Prévision Prophet du retard moyen journalier -- {key.replace('_', ' ')}")
ax.set_ylabel("Retard moyen (min)"); ax.legend(); plt.tight_layout(); plt.show()

fig2 = pm.plot_components(fc)
fig2.suptitle(f"Composantes saisonnières -- {key.replace('_', ' ')}", fontsize=11)
plt.tight_layout(); plt.show()

---
## 2. Module 2 -- Repli GPS

### 2.1 Charger le modèle et une journée de bus réelle

In [62]:
from src.models import gps_fallback as fb_mod
from src.data import fallback as fb
from src.data import foundation as fdn
from src.data.db import get_db

models_fb   = fb_mod.load(MODELS_DIR / "fallback")
db_winicari = get_db("winicari")
db_gps      = get_db("Historique_pos")
cfg_fdn     = fdn.Config()
usable      = fdn.build_usable_lines(db_winicari, cfg_fdn)

LINE, SOCIETE = "209", "S.R.T.K"
stops = usable[(LINE, SOCIETE)]

# Auto-discover most recent day with pings for this line
day_col, bus_id = None, None
for dc in sorted(db_gps.list_collection_names(), reverse=True):
    if not dc.startswith("d"): continue
    sample = db_gps[dc].distinct("bus.code", {"service.codeLigne": LINE})
    if sample:
        day_col = dc
        bus_id  = int(sample[0])
        break

print(f"Using day={day_col}  bus={bus_id}")
raw   = fdn.load_pings(db_gps, day_col, LINE, bus_id)
g     = fdn.clean_pings(raw, cfg_fdn)
g, route_len = fdn.project_to_route(g, stops, cfg_fdn)
g_kf  = fb_mod.run_kalman(g, route_len)
gaps  = fb.gap_table(g_kf)

print(f"Pings: {len(g_kf):,}  |  Route: {route_len/1000:.1f} km  |  Signal gaps: {len(gaps)}")
if len(gaps):
    display(gaps[["gap_min","s_start_km","s_end_km","dist_covered_km","speed_before_kph"]].head())


GPS Fallback LSTM correction loaded  (window=10, trained on 3,929 pings)
Using day=d20260618  bus=6030
Pings: 3,931  |  Route: 192.0 km  |  Signal gaps: 2


,gap_min,s_start_km,s_end_km,dist_covered_km,speed_before_kph
0,31.2,56.0,93.8,37.8,99.3
1,31.5,104.9,116.4,11.5,96.5


### 2.2 Évaluation sur écarts synthétiques -- interp vs Kalman vs Kalman+LSTM

In [63]:
# Use first trip segment (or full day if no segments found)
segs = fdn.segment_trips(g_kf, route_len, cfg_fdn)
if len(segs) > 0:
    seg = segs.iloc[0]
    t_arr = pd.to_datetime(g_kf["t"])
    g_trip = g_kf[(t_arr >= seg["start"]) & (t_arr <= seg["end"])].reset_index(drop=True)
    g_trip = fb_mod.run_kalman(g_trip, route_len)
else:
    g_trip = g_kf.reset_index(drop=True)

print(f"Trip segment: {len(g_trip)} pings")

MASK_S = 3 * 60
rng    = np.random.default_rng(42)
t_unix = pd.to_datetime(g_trip["t"]).astype(np.int64).values // 10**9
cands  = np.where(~g_trip["signal_gap"].values)[0]
cands  = cands[(cands > 5) & (cands < len(g_trip) - 5)]

results = []
for _ in range(400):
    if len(cands) == 0: break
    i0 = int(rng.choice(cands))
    future = np.where(t_unix > t_unix[i0] + MASK_S)[0]
    if not len(future): continue
    i1 = int(future[0])
    inside = g_trip.iloc[i0+1:i1]
    if not len(inside): continue
    mid  = inside.iloc[len(inside)//2]
    t_q  = pd.Timestamp(mid["t"])
    tlat, tlon = float(mid["lat"]), float(mid["lon"])
    b4, af     = g_trip.iloc[i0], g_trip.iloc[i1]

    li, lo, _ = fb.interp_position(t_q,
                    pd.Timestamp(b4["t"]), float(b4["s"]),
                    pd.Timestamp(af["t"]), float(af["s"]), stops)
    kr  = fb.kalman_fallback(g_trip, t_q, stops)
    klr = fb_mod.predict_position(models_fb, g_trip, t_q, stops)

    results.append({
        "err_interp_m": fb.haversine_m(tlat, tlon, li, lo),
        "err_kalman_m": fb.haversine_m(tlat, tlon, kr["lat"], kr["lon"]) if kr else np.nan,
        "err_klstm_m":  fb.haversine_m(tlat, tlon, klr["lat"], klr["lon"]) if klr else np.nan,
        "gap_duration_s": (pd.Timestamp(af["t"]) - pd.Timestamp(b4["t"])).total_seconds(),
    })

res = pd.DataFrame(results).dropna()
print(f"Evaluated {len(res)} synthetic gaps\n")

summary = pd.DataFrame({
    "Method":          ["Linear interpolation", "Kalman filter", "Kalman + LSTM"],
    "Median error (m)":[res["err_interp_m"].median(), res["err_kalman_m"].median(),
                        res["err_klstm_m"].median()],
    "P90 error (m)":   [res["err_interp_m"].quantile(0.9), res["err_kalman_m"].quantile(0.9),
                        res["err_klstm_m"].quantile(0.9)],
    "Mean error (m)":  [res["err_interp_m"].mean(), res["err_kalman_m"].mean(),
                        res["err_klstm_m"].mean()],
}).round(0)
display(summary)


Trip segment: 1877 pings
Evaluated 395 synthetic gaps



,Method,Median error (m),P90 error (m),Mean error (m)
0,Linear interpolation,371.0,828.0,475.0
1,Kalman filter,159.0,692.0,346.0
2,Kalman + LSTM,159.0,692.0,344.0


### 2.3 Graphiques d'erreur -- distributions et erreur vs durée d'écart

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Repli GPS -- Évaluation sur écarts synthétiques", fontweight="bold")

cols = ["err_interp_m", "err_kalman_m", "err_klstm_m"]
labs = ["Interp.\nlinéaire", "Kalman", "Kalman\n+LSTM"]
clrs = ["#a6cee3", "#1f78b4", "#b2df8a"]

bp = axes[0].boxplot([res[c].dropna() for c in cols], patch_artist=True,
                     medianprops={"color":"black","lw":2},
                     flierprops={"marker":".", "alpha":0.3})
for patch, c in zip(bp["boxes"], clrs): patch.set_facecolor(c)
axes[0].set_xticklabels(labs)
axes[0].set_ylabel("Erreur de position (m)")
axes[0].set_title("A -- Distribution des erreurs (boîte à moustaches)")

bins = np.linspace(0, 2000, 50)
for col, lab, c in zip(cols, ["Interp","Kalman","K+LSTM"], clrs):
    axes[1].hist(res[col].dropna(), bins=bins, alpha=0.5, label=lab, color=c)
    axes[1].axvline(res[col].median(), color=c, ls="--", lw=1.5)
axes[1].set_xlabel("Erreur (m)"); axes[1].set_ylabel("Nombre")
axes[1].set_title("B -- Histogrammes des erreurs (tirets = médiane)")
axes[1].legend()

axes[2].scatter(res["gap_duration_s"]/60, res["err_interp_m"],
                alpha=0.3, s=12, color="#a6cee3", label="Interp")
axes[2].scatter(res["gap_duration_s"]/60, res["err_klstm_m"],
                alpha=0.3, s=12, color="#b2df8a", label="K+LSTM")
axes[2].set_xlabel("Durée de l'écart (min)"); axes[2].set_ylabel("Erreur (m)")
axes[2].set_title("C -- Erreur vs durée de l'écart")
axes[2].legend()

plt.tight_layout(); plt.show()

### 2.4 Bande d'incertitude Kalman sur la journée complète

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

axes[0].plot(g_kf["t"], g_kf["s"]/1000,  lw=0.5, color="grey", alpha=0.5, label="GPS brut s")
axes[0].plot(g_kf["t"], g_kf["ks"]/1000, lw=1.2, color="navy", label="Kalman s")
for _, row in gaps.iterrows():
    axes[0].axvspan(row["t_start"], row["t_end"], alpha=0.15, color="red")
axes[0].set_ylabel("Distance sur l'itinéraire (km)")
axes[0].set_title(f"Bus {bus_id} -- ligne {LINE} -- {day_col}  (rouge = écarts de signal)")
axes[0].legend(fontsize=9)

axes[1].fill_between(g_kf["t"],
    (g_kf["ks"] - g_kf["kp"])/1000,
    (g_kf["ks"] + g_kf["kp"])/1000,
    alpha=0.3, color="orange", label="+-1 std incertitude")
axes[1].plot(g_kf["t"], g_kf["ks"]/1000, lw=1, color="navy")
for _, row in gaps.iterrows():
    axes[1].axvspan(row["t_start"], row["t_end"], alpha=0.15, color="red")
axes[1].set_ylabel("Distance sur l'itinéraire (km)"); axes[1].set_xlabel("Heure")
axes[1].set_title("L'incertitude grandit pendant les écarts, s'effondre à la récupération GPS")
axes[1].legend(fontsize=9)

plt.tight_layout(); plt.show()

---
## 3. Module 3 -- Détection d'Anomalies

### 3.1 Charger les modèles et les trajets scorés

In [66]:
from src.models import anomaly as anom_mod
from src.data import anomaly as an

models_anom = anom_mod.load(MODELS_DIR / "anomaly")
trips = models_anom["trips"].copy()

has_lstm = "lstm_score" in trips.columns

print(f"Total trips scored : {len(trips):,}")
print(f"IF flagged         : {trips['anomaly'].sum():,}  ({100*trips['anomaly'].mean():.1f}%)")
if has_lstm:
    print(f"LSTM flagged       : {trips['lstm_anomaly'].sum():,}  ({100*trips['lstm_anomaly'].mean():.1f}%)")
    print(f"Dual flagged       : {trips['dual_anomaly'].sum():,}")
else:
    print("(LSTM scores not in parquet -- re-run train_pipeline to add them)")


Anomaly models loaded (IF + LSTM AE, threshold=0.05503)
Total trips scored : 17,565
IF flagged         : 879  (5.0%)
LSTM flagged       : 793  (4.5%)
Dual flagged       : 54


### 3.2 Distributions des scores avec seuils de détection

In [ ]:
ncols = 2 if has_lstm else 1
fig, axes = plt.subplots(1, ncols, figsize=(7*ncols, 5))
if ncols == 1: axes = [axes]
fig.suptitle("Détection d'Anomalies -- Distributions des Scores", fontweight="bold")

ax = axes[0]
ax.hist(trips[~trips["anomaly"]]["if_score"], bins=60, alpha=0.7, color="#74c476",
        label=f"Normal ({(~trips['anomaly']).sum():,})")
ax.hist(trips[ trips["anomaly"]]["if_score"], bins=60, alpha=0.7, color="#fc8d59",
        label=f"Anomalie ({trips['anomaly'].sum():,})")
ax.axvline(0, color="black", ls="--", lw=1.5, label="Seuil IF (0)")
ax.set_title("A -- Score Isolation Forest")
ax.set_xlabel("Score IF (plus négatif = plus anormal)")
ax.set_ylabel("Nombre"); ax.legend()

if has_lstm:
    ax = axes[1]
    thr = models_anom["threshold"]
    ax.hist(trips[~trips["lstm_anomaly"]]["lstm_score"], bins=60, alpha=0.7, color="#74c476",
            label=f"Normal ({(~trips['lstm_anomaly']).sum():,})")
    ax.hist(trips[ trips["lstm_anomaly"]]["lstm_score"], bins=60, alpha=0.7, color="#fc8d59",
            label=f"Anomalie ({trips['lstm_anomaly'].sum():,})")
    ax.axvline(thr, color="black", ls="--", lw=1.5,
               label=f"Seuil LSTM (P95={thr:.4f})")
    ax.set_title("B -- Erreur de reconstruction de l'Autoencodeur LSTM")
    ax.set_xlabel("Erreur de reconstruction (MSE)"); ax.legend()

plt.tight_layout(); plt.show()

### 3.3 Score IF vs score LSTM -- accord entre les modèles

In [ ]:
if has_lstm:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    groups = [
        (~trips["anomaly"] & ~trips["lstm_anomaly"], "Les deux normaux",  "#74c476"),
        ( trips["anomaly"] & ~trips["lstm_anomaly"], "IF seulement",      "#fdae61"),
        (~trips["anomaly"] &  trips["lstm_anomaly"], "LSTM seulement",    "#abd9e9"),
        ( trips["anomaly"] &  trips["lstm_anomaly"], "Doubles (les deux)","#d7191c"),
    ]
    for mask, label, color in groups:
        ax.scatter(trips[mask]["if_score"], trips[mask]["lstm_score"],
                   alpha=0.3, s=10, color=color, label=f"{label} ({mask.sum():,})")
    ax.axvline(0, color="black", ls="--", lw=0.8)
    ax.axhline(models_anom["threshold"], color="black", ls="--", lw=0.8)
    ax.set_xlabel("Score IF"); ax.set_ylabel("Erreur de reconstruction LSTM")
    ax.set_title("A -- Score IF vs LSTM (anomalies doubles en rouge)")
    ax.legend(fontsize=8, markerscale=2)

    ax = axes[1]
    vals = [(~trips["anomaly"] & ~trips["lstm_anomaly"]).sum(),
            ( trips["anomaly"] & ~trips["lstm_anomaly"]).sum(),
            (~trips["anomaly"] &  trips["lstm_anomaly"]).sum(),
            ( trips["anomaly"] &  trips["lstm_anomaly"]).sum()]
    labs = [f"Les deux normaux\n({vals[0]:,})", f"IF seulement\n({vals[1]:,})",
            f"LSTM seulement\n({vals[2]:,})", f"Doubles\n({vals[3]:,})"]
    ax.pie(vals, labels=labs, colors=["#74c476","#fdae61","#abd9e9","#d7191c"],
           autopct="%1.1f%%", startangle=140, textprops={"fontsize":9})
    ax.set_title("B -- Répartition de l'accord entre modèles")
    plt.tight_layout(); plt.show()
else:
    print("Scores LSTM non disponibles -- relancer train_pipeline pour les inclure.")

### 3.4 Comparaison des caractéristiques -- trajets normaux vs anormaux

In [ ]:
features = an.FEATURES
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("Distributions des caractéristiques : Normal vs Anormal (Isolation Forest)", fontweight="bold")

for ax, feat in zip(axes.flat, features):
    q01, q99 = trips[feat].quantile(0.01), trips[feat].quantile(0.99)
    n_v = trips[~trips["anomaly"]][feat].clip(q01, q99)
    a_v = trips[ trips["anomaly"]][feat].clip(q01, q99)
    bins = np.linspace(min(n_v.min(), a_v.min()), max(n_v.max(), a_v.max()), 40)
    ax.hist(n_v, bins=bins, alpha=0.6, color="#74c476",
            label=f"Normal (méd={n_v.median():.1f})", density=True)
    ax.hist(a_v, bins=bins, alpha=0.7, color="#fc8d59",
            label=f"Anomalie (méd={a_v.median():.1f})", density=True)
    ax.set_title(feat); ax.legend(fontsize=7)

plt.tight_layout(); plt.show()

### 3.5 Taux d'anomalie par compagnie et dans le temps

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

by_soc = (trips.groupby("societe")
               .agg(total=("anomaly","count"), flagged=("anomaly","sum"))
               .assign(rate=lambda x: 100*x["flagged"]/x["total"])
               .sort_values("rate"))
axes[0].barh(by_soc.index, by_soc["rate"], color="#fc8d59", edgecolor="white")
for i, (idx, row) in enumerate(by_soc.iterrows()):
    axes[0].text(row["rate"]+0.1, i, f'{int(row["flagged"])}/{int(row["total"])}',
                 va="center", fontsize=8)
axes[0].set_xlabel("Taux d'anomalie (%)")
axes[0].set_title("A -- Taux d'anomalie par compagnie (Isolation Forest)")

trips["date"] = pd.to_datetime(trips["day"].astype(str), format="%Y%m%d")
daily = trips.groupby("date")["anomaly"].sum().reset_index()
daily["roll7"] = daily["anomaly"].rolling(7, center=True).mean()
axes[1].bar(daily["date"], daily["anomaly"], color="#fc8d59", alpha=0.4, label="Comptage journalier")
axes[1].plot(daily["date"], daily["roll7"], color="darkred", lw=1.5, label="Moyenne glissante 7 jours")
axes[1].set_xlabel("Date"); axes[1].set_ylabel("Trajets anormaux")
axes[1].set_title("B -- Comptage quotidien d'anomalies dans le temps"); axes[1].legend()

plt.tight_layout(); plt.show()

### 3.6 Top 10 des trajets les plus anormaux

In [71]:
show_cols = [c for c in ["day","line","societe","dir","bus","n_stops","match_rate",
             "max_dwell_s","total_elapsed","if_score","lstm_score"] if c in trips.columns]
top10 = trips.sort_values("if_score").head(10)[show_cols].reset_index(drop=True)
top10["if_score"]   = top10["if_score"].round(3)
top10["match_rate"] = top10["match_rate"].round(2)
if "lstm_score" in top10: top10["lstm_score"] = top10["lstm_score"].round(5)
print("Top 10 most anomalous trips (lowest IF score):")
display(top10)


Top 10 most anomalous trips (lowest IF score):


,day,line,societe,dir,bus,n_stops,match_rate,max_dwell_s,total_elapsed,if_score,lstm_score
0,20250520,306,S.R.T.K,RETOUR,6029,8,1.0,11400.1,626.172850,-0.776,0.00019
1,20250718,304,S.T.S,RETOUR,788,27,1.0,8610.3,475.964150,-0.773,0.00008
2,20251112,215,S.R.T.K,RETOUR,6029,12,1.0,9891.9,366.847283,-0.771,0.00011
3,20260620,306,S.T.S,ALLER,111,25,1.0,12716.7,305.168917,-0.769,0.00010
4,20260213,215,S.R.T.K,RETOUR,6029,14,1.0,9703.7,467.329900,-0.767,0.04453
5,20250419,209,S.R.T.K,RETOUR,6028,3,1.0,10798.9,319.993300,-0.762,0.00018
6,20260326,215,S.R.T.K,RETOUR,6029,9,1.0,7029.6,413.015983,-0.760,0.00007
7,20251001,212,S.R.T.K,RETOUR,6040,4,1.0,6187.8,326.464700,-0.756,0.00005
8,20260401,215,S.R.T.K,RETOUR,6029,18,1.0,7459.4,404.410333,-0.755,0.00522
9,20260317,215,S.R.T.K,RETOUR,6029,6,1.0,8463.9,353.901300,-0.755,0.01005


---
## 4. Module 4 -- Chatbot RAG

### 4.1 Charger la base de connaissances et lancer les requêtes de test

In [72]:
from src.models import chatbot as chat_mod

CHROMA_DIR = Path("../data/processed/chroma_kb")
models_chat = chat_mod.load(CHROMA_DIR)
print(f"Knowledge base: {models_chat['col'].count()} documents indexed")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Knowledge base loaded: 1276 documents
Knowledge base: 1276 documents indexed


In [73]:
TEST_QUERIES = [
    "Which line has the highest number of anomalous trips?",
    "What is the GPS stop coverage quality of line 209?",
    "Which company has the worst average GPS match rate?",
    "Are there any lines with no geocoded stops?",
    "What was the longest stop dwell time recorded in the data?",
    "How many service days does S.R.T.K operate on average?",
]

rag_results = []
for q in TEST_QUERIES:
    try:
        r = chat_mod.ask(models_chat, q)
        rag_results.append({
            "Question": q,
            "Answer": r["answer"],
            "Tokens": r.get("tokens_used", 0),
            "Docs": len(r.get("context", [])),
        })
    except Exception as e:
        rag_results.append({"Question": q, "Answer": f"ERROR: {e}", "Tokens": 0, "Docs": 0})

for i, row in enumerate(rag_results, 1):
    print(f"Q{i}: {row['Question']}")
    print(f"A:  {str(row['Answer'])[:400]}")
    print(f"    [{row['Tokens']} tokens | {row['Docs']} docs]\n")


Q1: Which line has the highest number of anomalous trips?
A:  Based on the provided context, I can see that there are anomalous trips detected for lines 212, 306, and 217. 

Line 306 has 2 anomalous trips (direction ALLER and direction RETOUR), and Line 217 also has 2 anomalous trips (direction ALLER on two different dates). 

Line 212 has 1 anomalous trip.

Therefore, lines 306 and 217 are tied for the highest number of anomalous trips, with 2 each.
    [524 tokens | 5 docs]

Q2: What is the GPS stop coverage quality of line 209?
A:  Line 209 has two instances in the data. 

1. Operated by S.T.S., it has partial GPS stop coverage (3/5 stops geocoded).
2. Operated by S.R.T.K., it has full GPS stop coverage (22/22 stops geocoded).

To provide a definitive answer, I need more information about which instance you are referring to.
    [398 tokens | 5 docs]

Q3: Which company has the worst average GPS match rate?
A:  Company S.R.T.K has the worst average GPS match rate at 66%.
    [392 tok

In [ ]:
rag_df = pd.DataFrame(rag_results)
print(f"Jetons totaux utilisés : {rag_df['Tokens'].sum():,}")
print(f"Moyenne par requête    : {rag_df['Tokens'].mean():.0f}")

fig, ax = plt.subplots(figsize=(9, 3))
ax.barh([f"Q{i+1}" for i in range(len(rag_df))], rag_df["Tokens"], color="#6baed6")
ax.set_xlabel("Jetons utilisés"); ax.set_title("RAG -- Consommation de jetons par requête")
plt.tight_layout(); plt.show()

---
## 5. Tableau de bord récapitulatif

In [75]:
print("=" * 60)
print("WiniCari AI -- Evaluation Scorecard")
print("=" * 60)

print("\n[Module 1] Delay Prediction")
hgbm_row = metric_df[metric_df["Model"]=="HistGBM"].iloc[0]
lstm_row  = metric_df[metric_df["Model"]=="LSTM"].iloc[0]
base_row  = metric_df[metric_df["Model"].str.contains("Persistence")].iloc[0]
print(f"  HistGBM  MAE={hgbm_row['MAE (min)']:.2f} min  RMSE={hgbm_row['RMSE (min)']:.2f}  R2={hgbm_row['R2']:.3f}")
print(f"  LSTM     MAE={lstm_row['MAE (min)']:.2f} min  RMSE={lstm_row['RMSE (min)']:.2f}  R2={lstm_row['R2']:.3f}")
print(f"  Baseline MAE={base_row['MAE (min)']:.2f} min  (persistence)")
print(f"  Prophet  {len(models_delay['prophet'])} line/direction models")

print("\n[Module 2] GPS Fallback (synthetic 3-min gaps)")
for label, col in [("Linear interp ","err_interp_m"),
                    ("Kalman        ","err_kalman_m"),
                    ("Kalman + LSTM ","err_klstm_m")]:
    med = res[col].median()
    p90 = res[col].quantile(0.9)
    print(f"  {label}: median={med:.0f} m  P90={p90:.0f} m")

print("\n[Module 3] Anomaly Detection")
print(f"  Isolation Forest : {int(trips['anomaly'].sum()):,}/{len(trips):,} trips  ({100*trips['anomaly'].mean():.1f}%)")
if has_lstm:
    print(f"  LSTM Autoencoder : {int(trips['lstm_anomaly'].sum()):,}/{len(trips):,} trips  ({100*trips['lstm_anomaly'].mean():.1f}%)")
    print(f"  Dual-flagged     : {int(trips['dual_anomaly'].sum()):,}")

print("\n[Module 4] RAG Chatbot")
print(f"  Knowledge base   : {models_chat['col'].count():,} documents")
print(f"  Test queries     : {len(rag_df)}  avg {rag_df['Tokens'].mean():.0f} tokens/query")
print("=" * 60)


WiniCari AI -- Evaluation Scorecard

[Module 1] Delay Prediction
  HistGBM  MAE=2.75 min  RMSE=6.82  R2=0.912
  LSTM     MAE=3.03 min  RMSE=7.12  R2=0.904
  Baseline MAE=3.06 min  (persistence)
  Prophet  33 line/direction models

[Module 2] GPS Fallback (synthetic 3-min gaps)
  Linear interp : median=371 m  P90=828 m
  Kalman        : median=159 m  P90=692 m
  Kalman + LSTM : median=159 m  P90=692 m

[Module 3] Anomaly Detection
  Isolation Forest : 879/17,565 trips  (5.0%)
  LSTM Autoencoder : 793/17,565 trips  (4.5%)
  Dual-flagged     : 54

[Module 4] RAG Chatbot
  Knowledge base   : 1,276 documents
  Test queries     : 6  avg 420 tokens/query
